In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FOLDERNAME = 'ML/Final/ML_final/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

%cd /content/drive/My\ Drive/$FOLDERNAME

Mounted at /content/drive
/content/drive/My Drive/ML/Final/ML_final


In [ ]:
##### LightGBM model notebook

import numpy as np
import pandas as pd

import os

In [ ]:
!pip install wandb -q
!pip install lightgbm


In [ ]:
%run data_extraction_feature_engineering.ipynb
df_train_full, df_test_full = extract_data()
VAL_START_DATE = '2012-09-01'
df_with_features = create_features1(df_train_full)
X_train, y_train, X_val, y_val = train_and_val_split(df_with_features, VAL_START_DATE, skip_dummies=True)


Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  .

In [ ]:

import wandb
import os

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
print(f"Holiday Weeks in Train: {X_train['IsHoliday'].sum()}")
print(f"Holiday Weeks in Val:   {X_val['IsHoliday'].sum()}")

Holiday Weeks in Train: 26695
Holiday Weeks in Val:   2966


In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import wandb
import os
import joblib

run = wandb.init(
    project="walmart-sales-forecasting_lightGBM",
    name="LightGBM_Baseline_WandB_Only",
    config={
        'objective': 'regression',
        'metric': 'mae',
        'learning_rate': 0.05,
        'max_depth': 6,          # Controls tree depth
        'num_leaves': 31,        # Specific to LightGBM: Max leaves per tree
        'n_estimators': 500,     # Max number of trees
        'early_stopping_rounds': 20,
        'random_state': 42,
        'subsample': 0.8,        # GOSS sampling rate
        'colsample_bytree': 0.8  # Feature sampling
    }
)

# ---------------------------------------------------------
# TRAIN LIGHTGBM MODEL
# ---------------------------------------------------------

# Initialize Model
model = lgb.LGBMRegressor(
    objective='regression',
    metric='mae',
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,       # Important: Should be < 2^max_depth
    n_estimators=500,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1           # Suppress training messages
)

try:
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=20),
            lgb.log_evaluation(period=0) # Disable periodic logging for cleanliness
        ]
    )
except Exception as e:
    print(f"Error during fitting: {e}")


# ---------------------------------------------------------
# FINAL EVALUATION & LOGGING
# ---------------------------------------------------------
y_pred = model.predict(X_val)
y_train_pred = model.predict(X_train)

# Get Holiday Status from original DataFrames (safer than X columns)
if 'IsHoliday' in X_val.columns:
    is_holiday = X_val['IsHoliday'].values
else:
    is_holiday = np.zeros(len(y_val))

if 'IsHoliday' in X_train.columns:
    is_holiday_train = X_train['IsHoliday'].values
else:
    is_holiday_train = np.zeros(len(y_train))

weights = np.where(is_holiday, 5, 1)
weights_train = np.where(is_holiday_train, 5, 1)

final_wmae = np.average(np.abs(y_val - y_pred), weights=weights)
final_mae = mean_absolute_error(y_val, y_pred)

train_wmae = np.average(np.abs(y_train - y_train_pred), weights=weights_train)
train_mae = mean_absolute_error(y_train, y_train_pred)

if run:
    # Log Final Metrics
    wandb.log({"final_validation_wmae": final_wmae, "final_validation_mae": final_mae})
    wandb.log({"train_wmae": train_wmae, "train_mae": train_mae})


    # --- FEATURE IMPORTANCE LOGGING FOR LIGHTGBM ---
    # LightGBM returns feature importance as a dict or array depending on version
    try:
        importance = model.feature_importances_
        feature_names = X_train.columns.tolist()

        # Ensure lengths match
        if len(importance) == len(feature_names):
            importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
            importance_df = importance_df.sort_values(by='importance', ascending=False)

            # Use wandb.Table
            table = wandb.Table(dataframe=importance_df.head(20))
            wandb.log({"feature_importance_table": table})

            # Plot
            import matplotlib.pyplot as plt
            plt.figure(figsize=(10, 6))
            plt.barh(importance_df.head(20)['feature'], importance_df.head(20)['importance'])
            plt.gca().invert_yaxis()
            plt.title("Top 20 Feature Importances (LightGBM)")
            plt.tight_layout()
            wandb.log({"feature_importance_plot": wandb.Image(plt)})
            plt.close()
        else:
            print(f"Warning: Mismatch in feature importance length ({len(importance)}) vs features ({len(feature_names)})")
    except Exception as e:
        print(f"Error logging feature importance: {e}")
    # ----------------------------------------

    # Save Model Artifact to W&B
    artifact = wandb.Artifact("lightgbm-model-final", type="model")
    joblib.dump(model, "model_lgb.joblib")
    artifact.add_file("model_lgb.joblib")
    run.log_artifact(artifact)

print(f"Final Train WMAE: {train_wmae:.2f}")
print(f"Final Train MAE: {train_mae:.2f}")

print(f"Final Validation WMAE: {final_wmae:.2f}")
print(f"Final Validation MAE: {final_mae:.2f}")

if run:
    wandb.finish()

import gc
gc.collect() # Forces Python to free up unused memory


Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[199]	valid_0's l1: 1252.14
Final Train WMAE: 1859.10
Final Train MAE: 1636.99
Final Validation WMAE: 1361.60
Final Validation MAE: 1252.14


final_validation_mae,▁
final_validation_wmae,▁
train_mae,▁
train_wmae,▁
final_validation_mae,1252.14378
final_validation_wmae,1361.60393
train_mae,1636.99198
train_wmae,1859.09594


346